# Touster — one-notebook LoRA fine-tuning

Fine-tune a small LLM end-to-end: dataset → search → export. Runs on a free Colab T4 or locally on CPU.

**Quickstart:** Runtime → Run all, then edit the **Config** cell (base model + dataset topic).

### 0 · Install
Light core only (no torch/transformers yet — those arrive with the tuning stage).

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'touster @ git+https://github.com/Patan-Sameer66/touster.git'])
print('Touster core installed.')

### 1 · Hardware check
Detects GPU/CPU/RAM, ranks models that fit. Read-only — note a model id for the Config cell below.

In [1]:
from touster.hardware.detect import detect_hardware
from touster.hardware.report import print_hardware_report

hw = detect_hardware()
print_hardware_report(hw)

,
Platform,CPU only
GPU,None (None)
System RAM,16.8 GB
CPU cores,14


#,Model,Params,VRAM(GB),t/s,Quality,Fits
1,smollm2-1.7b,1.7B,1.6,-,65,OK
2,llama3.2-1b,1.2B,1.3,-,60,OK
3,qwen2.5-0.5b,0.5B,0.8,-,55,OK
4,tiny-gpt2,0.1B,0.6,-,5,OK


Top suggestion: smollm2-1.7b (enter a model id above, or press Enter to accept)


'HuggingFaceTB/SmolLM2-1.7B-Instruct'

### 2 · Config — *the only cell you normally edit*
Base model, dataset mode (0 generate / 1 structure / 2 bring-your-own), and the LLM used for generation + quality judging.

In [ ]:
from pathlib import Path
from touster.dataset.modes import prepare_dataset_stage
from touster.tuning.prepare import prepare_tuning_stage

# ═══════════════════════ BASE MODEL (from hardware table) ═══════════════════
BASE_MODEL = 'HuggingFaceTB/SmolLM2-1.7B-Instruct'

# ═══════════════════════════════ DATASET ═════════════════════════════════
DATASET_MODE  = 0          # 0 = generate, 1 = structure raw text, 2 = bring your own
DATASET_TOPIC = 'Python coding tips and tricks'  # mode 0
DATASET_PATH  = None       # mode 1 (raw .txt/.md) or mode 2 (file / URL / HF id)
NUM_SAMPLES   = 200        # samples to generate/keep
GEN_BATCH_SIZE = 10        # Q&A pairs per LLM call; lower if you see JSON errors
MIN_QUALITY_SCORE = 3.0    # quality-judge threshold, scale is 1-QUALITY_SCALE
QUALITY_SCALE = 5

# ═══════════════════ LLM CLIENT (generation + quality judge) ═══════════════
API_KEY, API_BASE, API_MODEL = '', '', ''   # leave blank to use Ollama instead
OLLAMA_PORT, OLLAMA_MODEL = 11434, 'qwen2.5:3b'

# ═══════════════════════════════ TUNING LOOP ═══════════════════════════════
MAX_TRIALS    = 20     # trials (CPU: 3-5; GPU: up to 20)
TRIAL_STEPS   = 50     # steps per trial
FINAL_STEPS   = 200    # steps for the final full training run
USE_LLM_PRIOR = True   # LLM narrows the search space once; Optuna's TPE sampler decides every trial
JUDGE_TOP_K   = 3       # LLM-judge only the top-k survivors — the expensive step
JUDGE_PROMPTS = 20

# ═══════════════ EXPORT (config-driven — runs automatically, not a stage) ═══════════════
SAVE_LOCAL     = True
LOCAL_SAVE_DIR = 'touster_out'
EXPORT_MERGED  = True
EXPORT_GGUF    = True
GGUF_QUANTIZE  = 'q4_k_m'
HF_TOKEN, HF_REPO_ID = '', ''   # leave blank to skip the HF Hub push

# ═══════════════════════════════ RUN DIRECTORY ═════════════════════════════
run_dir = Path('touster_run')
run_dir.mkdir(parents=True, exist_ok=True)

# ── everything below is package logic, not config — see touster/dataset/modes.py
# and touster/tuning/prepare.py
dataset_cfg, validated_path, llm_client = prepare_dataset_stage(
    BASE_MODEL, DATASET_MODE, DATASET_TOPIC, DATASET_PATH, NUM_SAMPLES, GEN_BATCH_SIZE,
    MIN_QUALITY_SCORE, QUALITY_SCALE, API_KEY, API_BASE, API_MODEL, OLLAMA_PORT, OLLAMA_MODEL,
)
recipe, loop_cfg, export_cfg = prepare_tuning_stage(
    BASE_MODEL, MAX_TRIALS, TRIAL_STEPS, FINAL_STEPS, USE_LLM_PRIOR, JUDGE_TOP_K, JUDGE_PROMPTS,
    dataset_cfg, SAVE_LOCAL, LOCAL_SAVE_DIR, EXPORT_MERGED, EXPORT_GGUF, GGUF_QUANTIZE, HF_TOKEN, HF_REPO_ID,
)

### 3 · Data source
Acquires (generate / structure / load) → dedup → LLM-judge quality gate → validate + repair → golden-format `dataset.jsonl`.

In [ ]:
from touster.dataset.modes import run_dataset_mode

if DATASET_MODE == 2 and validated_path is not None:
    print(f'Skipping generation (mode 2, local file) — using: {validated_path}')
    dataset_path = validated_path
else:
    dataset_path = run_dataset_mode(dataset_cfg, run_dir, llm_client)

print(f'Dataset: {dataset_path}')

### 3b · Install training deps
Heavy deps (torch/peft/transformers/optuna) — deferred until here since stages 1-3 never need them.

In [ ]:
import sys, subprocess

_extra = {'cuda': 'gpu', 'mlx': 'mlx', 'cpu': 'cpu'}[hw.platform]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                f'touster[{_extra}] @ git+https://github.com/Patan-Sameer66/touster.git'])
print(f'Training deps installed: touster[{_extra}]')

### 4 · Tuning
Optuna TPE search — fixed-budget trials, checkpointed, LLM-judge on top-k survivors, falls back to the default recipe if every trial fails. Trains the winner to completion.

In [ ]:
from touster.tuning.loop import run_loop

best_recipe, adapter_path = run_loop(recipe, loop_cfg, dataset_path, run_dir, llm_client)
print(f'Adapter : {adapter_path}')
print(f'Recipe  : lr={best_recipe.learning_rate:.2e} rank={best_recipe.lora_rank} scheduler={best_recipe.scheduler}')

### 4b · Export
Runs the config cell's export toggles automatically — local save, merged weights, GGUF, model card, optional HF Hub push. Never fails partway silently — one export failing doesn't block the others.

In [ ]:
from touster.tuning.export_stage import run_export_stage

export_results = run_export_stage(best_recipe, adapter_path, run_dir, export_cfg)
for name, path in export_results.items():
    print(f'{name:<12} {path if path else "skipped"}')